# Combining Models: Voting, Averaging, and Stacking

A single model only ever sees the data through its own assumptions, so it makes its own characteristic mistakes. *Ensemble methods* combine several models so that their individual errors partly cancel out, and the combined prediction is usually steadier and more accurate than any single member. In this notebook you work through four ways to combine classifiers on a medical dataset, from a simple majority vote to a trained meta-model.

## Learning Objectives

At the end of this notebook, you should be able to:

- Build a hard-voting and a soft-voting ensemble with scikit-learn's `VotingClassifier`.
- Combine model probabilities using simple averaging and weighted averaging.
- Construct a stacking classifier, both with `StackingClassifier` and from its component steps.
- Evaluate an ensemble with accuracy, F1, and recall, and explain what the scores mean.

## Why Combine Models?

The idea behind ensembles is the *wisdom of the crowd*: if several independent voters are each a little better than chance, the majority decision is more reliable than any single voter. The same holds for models, but only when two conditions are met:

1. **Each model is better than random guessing.** Combining weak but useful models helps; combining models that are no better than a coin flip does not.
2. **The models make _different_ mistakes.** If every model errs on the same rows, averaging them changes nothing. Diversity (different algorithms, different features, or different data subsets) is what lets the errors cancel.

This notebook builds up four combination strategies, in order of how much they learn about combining:

| Strategy | How the members are combined | What is learned |
|---|---|---|
| Max voting | Majority vote on the predicted labels | Nothing (fixed rule) |
| Averaging | Mean of the predicted probabilities | Nothing (fixed rule) |
| Weighted averaging | Weighted mean of the probabilities | Fixed weights you choose |
| Stacking | A second model is trained on the members' predictions | The combination itself |

## Setup

### Import Libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score

import warnings

warnings.filterwarnings("ignore")

RSEED = 42

### Load and Split the Data

We use the Pima Native Americans diabetes dataset. The file has no header row, so we pass `header=None` and refer to columns by index: column `8` is the target (1 = diabetes, 0 = no diabetes) and columns `0` to `7` are the features.

We split with `stratify=y` so the train and test sets keep the same class proportions as the full dataset. This matters here because the classes are imbalanced, and a non-stratified split could leave one set with too few positive cases.

In [ ]:
# Import diabetes data
df = pd.read_csv("data/pima-native-americans-diabetes.csv", header=None)
df.head(2)

In [ ]:
# Define features and target and split into train and test set
y = df[8]
X = df.drop(8, axis=1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=RSEED
)

In [ ]:
y_train.shape, y_test.shape

In [ ]:
y_train.value_counts(), y_test.value_counts()

The split keeps the imbalance intact: roughly two thirds of the patients are negative (no diabetes) and one third positive. Keep this in mind when reading accuracy later, because a model that always predicted "no diabetes" would already score about 65%.

### Baseline: How Good Is Each Model Alone?

Before combining anything, we measure each model on its own. These baselines are the bar the ensembles need to clear: an ensemble is only worthwhile if it beats its best single member.

In [ ]:
# Train each model on its own and compare test accuracy
baselines = {
    "Logistic Regression": LogisticRegression(random_state=RSEED),
    "K-Nearest Neighbours": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=RSEED),
    "Random Forest": RandomForestClassifier(random_state=RSEED),
}

for name, single_model in baselines.items():
    single_model.fit(X_train, y_train)
    print(f"{name:22s} test accuracy: {single_model.score(X_test, y_test):.3f}")

The four models land between roughly 0.69 and 0.75 on the test set, with the Random Forest strongest at about 0.75. None is clearly dominant, and they reach their decisions in very different ways, so they are good candidates to combine.

## Max Voting

In *max (hard) voting*, each model predicts a class label and the ensemble returns the label that most models chose, like a show of hands. It is the simplest ensemble: no probabilities, no weights, just a majority. We combine a Logistic Regression, a K-Nearest Neighbours model, and a Decision Tree.

```mermaid
flowchart TD
    D["Training data"] --> M1["Model 1<br/>Logistic Regression"]
    D --> M2["Model 2<br/>KNN"]
    D --> M3["Model 3<br/>Decision Tree"]
    M1 -->|"prediction"| C["Combine by a fixed rule<br/>(majority vote or average)"]
    M2 -->|"prediction"| C
    M3 -->|"prediction"| C
    C --> P["Final prediction"]
```

In [ ]:
from sklearn.ensemble import VotingClassifier

model1 = LogisticRegression(random_state=RSEED)
model2 = KNeighborsClassifier()
model3 = DecisionTreeClassifier(random_state=RSEED)

model = VotingClassifier(
    estimators=[("lr", model1), ("knn", model2), ("dt", model3)], voting="hard"
)
model.fit(X_train, y_train)
model.score(X_test, y_test)

This gives about 0.75, a small lift over each of the three voters on its own (0.69 to 0.745). Hard voting throws away how *confident* each model was, though: a model that was barely above 50% counts exactly as much as one that was certain. Soft voting addresses that.

### Soft Voting

*Soft voting* averages the predicted probabilities instead of counting labels, then picks the class with the highest average probability. A confident model therefore pulls the result more than an unsure one. Setting `voting="soft"` is the only change.

In [ ]:
# Soft voting: average the predicted probabilities instead of the labels
soft_model = VotingClassifier(
    estimators=[("lr", model1), ("knn", model2), ("dt", model3)], voting="soft"
)
soft_model.fit(X_train, y_train)
soft_model.score(X_test, y_test)

Soft voting reaches about 0.766, above hard voting, because it uses the extra information in the probabilities rather than collapsing each model to a single label.

## Averaging

We can also average probabilities by hand, without the `VotingClassifier`. Each model produces a probability for each class; we take the mean across models and predict the class with the highest mean probability. This is the manual version of soft voting and makes the mechanics explicit.

In [ ]:
model1 = LogisticRegression(random_state=RSEED)
model2 = KNeighborsClassifier()
model3 = DecisionTreeClassifier(random_state=RSEED)

model1.fit(X_train, y_train)
model2.fit(X_train, y_train)
model3.fit(X_train, y_train)

pred1 = model1.predict_proba(X_test)
pred2 = model2.predict_proba(X_test)
pred3 = model3.predict_proba(X_test)

finalpred = (pred1 + pred2 + pred3) / 3
finalpred = np.argmax(finalpred.round(0), axis=1)
(y_test == finalpred).sum() / len(finalpred)

Averaging reaches about 0.766, beating max voting (0.75) and every single model. Pooling the probabilities smooths out the individual models' overconfident mistakes.

## Weighted Averaging

Plain averaging treats every model as equally trustworthy. *Weighted averaging* gives better models a larger say. Here the weights are each model's training accuracy, normalised to sum to one.

> **Caveat:** weighting by *training* accuracy rewards models that overfit the training set (a Decision Tree can reach near-perfect training accuracy without generalising). In practice, weight by performance on a separate validation set, or tune the weights, rather than trusting training scores.

In [ ]:
model1 = LogisticRegression(random_state=RSEED)
model2 = KNeighborsClassifier()
model3 = DecisionTreeClassifier(random_state=RSEED)

model1.fit(X_train, y_train)
model2.fit(X_train, y_train)
model3.fit(X_train, y_train)

pred1 = model1.predict_proba(X_test)
pred2 = model2.predict_proba(X_test)
pred3 = model3.predict_proba(X_test)

acc1 = accuracy_score(y_train, model1.predict(X_train))
acc2 = accuracy_score(y_train, model2.predict(X_train))
acc3 = accuracy_score(y_train, model3.predict(X_train))

acc_sum = acc1 + acc2 + acc3

weight1 = acc1 / acc_sum
weight2 = acc2 / acc_sum
weight3 = acc3 / acc_sum

finalpred = pred1 * weight1 + pred2 * weight2 + pred3 * weight3
finalpred = np.argmax(finalpred.round(0), axis=1)
(y_test == finalpred).sum() / len(finalpred)

The score is about 0.766, essentially the same as plain averaging, because the three training accuracies are close, so the weights come out nearly equal. Weighting helps most when the models differ clearly in quality.

## Stacking

Voting and averaging use a *fixed* rule to combine models. *Stacking* goes a step further: it trains a second model, the **meta-model**, to learn the best way to combine the base models' predictions. Each base model outputs predictions; those predictions become the input features for the meta-model, which produces the final answer.

```mermaid
flowchart TD
    D["Training data"] --> M1["Base model 1<br/>Decision Tree"]
    D --> M2["Base model 2<br/>KNN"]
    D --> M3["Base model 3<br/>Random Forest"]
    M1 -->|"predicted probabilities"| MM["Meta-model<br/>Logistic Regression"]
    M2 -->|"predicted probabilities"| MM
    M3 -->|"predicted probabilities"| MM
    MM --> P["Final prediction"]
```

### Stacking with `StackingClassifier`

Scikit-learn's `StackingClassifier` wraps the whole process. You pass the base estimators and the `final_estimator` (the meta-model), and it trains the layers for you. Internally it uses cross-validation so the meta-model learns from out-of-fold predictions, not from predictions the base models have already seen during their own training.

In [ ]:
# Implementation of Stacking in Scikit-Learn
from sklearn.ensemble import StackingClassifier

estimators = [
    ("dt", DecisionTreeClassifier(random_state=RSEED)),
    ("knn", KNeighborsClassifier()),
    ("rf", RandomForestClassifier(random_state=RSEED)),
]

clf = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression())
clf.fit(X_train, y_train).score(X_test, y_test)

In order to simplify the example above, the stacking model we have created has only two levels. The **DecisionTree, KNN and RandomForest** models are built at **level zero**, while a **LogisticRegression** model is built at **level one**.

 

Here stacking scores about 0.745. With only three base models on a small dataset it does not beat simple averaging, which is common: stacking earns its keep when there are many diverse base models and enough data for the meta-model to learn a useful combination.

### Stacking by Hand

To see what `StackingClassifier` does internally, we build the same thing step by step. The key concern is **data leakage**: the meta-model must learn from predictions made on data the base models did *not* train on, otherwise it learns from over-optimistic predictions and overfits. We therefore split the training set into two halves: the base models train on the first half, and we collect their predictions on the second half to train the meta-model.

We take three base classifiers and one final estimator separately, and initialise the object for each one.

In [ ]:
# Base Estimators
dt = DecisionTreeClassifier(random_state=RSEED)
knn = KNeighborsClassifier()
rf = RandomForestClassifier(random_state=RSEED)

# final estimator
final_est = LogisticRegression()

For stacking classification we divide our train dataset into two parts
1. With the first part, we train our base estimators  
2. And with the second part we predict probabilities from base estimator and train the final estimator on the probabilities  

In [ ]:
X_train_1, X_train_2, y_train_1, y_train_2 = train_test_split(
    X_train, y_train, stratify=y_train, random_state=RSEED
)

In [ ]:
# Fit all the base estimators on the 1st half of the train dataset
dt_model = dt.fit(X_train_1, y_train_1)
knn_model = knn.fit(X_train_1, y_train_1)
rf_model = rf.fit(X_train_1, y_train_1)

# Then with the second half of the train dataset we predict the probabilities from the base estimators
dt_probab = dt_model.predict_proba(X_train_2)[:, 1]
knn_probab = knn_model.predict_proba(X_train_2)[:, 1]
rf_probab = rf_model.predict_proba(X_train_2)[:, 1]

We now have, for each patient in the second half, three numbers: the probability of diabetes according to each base model. Those three columns become the training features for the meta-model.

In [ ]:
# Then we combine all the probabilities and form a training data (probabilities) for the final estimator
lr_X = pd.concat(
    [
        pd.DataFrame(dt_probab, columns=["dt"]),
        pd.DataFrame(knn_probab, columns=["knn"]),
        pd.DataFrame(rf_probab, columns=["rf"]),
    ],
    axis=1,
)

In [ ]:
lr_X

In [ ]:
# Fit the final estimator on the combined probabilities and target values
final_est.fit(lr_X, y_train_2)

For the test dataset, we do the same thing as while training
1. predict probabilities from base estimators on the test dataset
2. combine the probabilities from base estimator to form a test dataset for final estimator
3. predict with final estimator on test data (probabilities)

In [ ]:
dt_pred = dt_model.predict_proba(X_test)[:, 1]
knn_pred = knn_model.predict_proba(X_test)[:, 1]
rf_pred = rf_model.predict_proba(X_test)[:, 1]

comb_pred = pd.concat(
    [
        pd.DataFrame(dt_pred, columns=["dt"]),
        pd.DataFrame(knn_pred, columns=["knn"]),
        pd.DataFrame(rf_pred, columns=["rf"]),
    ],
    axis=1,
)

pred_final = final_est.predict(comb_pred)

Finally we score the hand-built stack. This time we look beyond accuracy, because the classes are imbalanced and accuracy alone can mislead.

In [ ]:
from sklearn.metrics import f1_score, recall_score

print(f"Accuracy: {accuracy_score(y_test, pred_final):.3f}")
print(f"F1 score: {f1_score(y_test, pred_final):.3f}")
print(f"Recall: {recall_score(y_test, pred_final):.3f}")

On the test set this gives about 0.77 accuracy, but an F1 of roughly 0.63 and a recall of about 0.57. The gap matters: a recall of 0.57 means the model misses around 43% of the patients who actually have diabetes. For a medical screen that is the costly error, so accuracy on its own paints too rosy a picture. Always read imbalanced results with a metric that focuses on the positive class.

## Summary

In this notebook you:

- Measured single-model baselines and used them as the bar for every ensemble.
- Built hard-voting and soft-voting ensembles with `VotingClassifier`.
- Combined predicted probabilities through plain and weighted averaging.
- Constructed a stacking classifier with `StackingClassifier` and reproduced it by hand.
- Compared the methods with accuracy, F1, and recall, and interpreted them on imbalanced data.

| Method | Test accuracy |
|---|---|
| Best single model (Random Forest) | 0.750 |
| Max voting | 0.750 |
| Soft voting | 0.766 |
| Averaging | 0.766 |
| Weighted averaging | 0.766 |
| Stacking (`StackingClassifier`) | 0.745 |

On this dataset, averaging the probabilities gave the largest lift, while stacking did not pay off with so few base models. The general lesson holds: ensembles help most when the members are individually decent and make different mistakes, and the right metric, not just accuracy, tells you whether the gain is real.

## References & Further Reading

- [**Scikit-learn: Ensemble Methods**](https://scikit-learn.org/stable/modules/ensemble.html): User guide for voting, stacking, bagging, and boosting.
- [**Scikit-learn: VotingClassifier**](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html): API reference for hard and soft voting.
- [**Scikit-learn: StackingClassifier**](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html): API reference for stacked generalisation.
- [**The Random Forest Algorithm (MLU-Explain)**](https://mlu-explain.github.io/random-forest/): A visual explanation of why combining many models works.